In [ ]:
import sagemaker
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker import get_execution_role
import time

# ─── Setup ─────────────────────────────────────────────────────
region = "me-south-1"
session = sagemaker.Session()
role = get_execution_role()
timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
job_name = f"explanation-job-hj-{timestamp}"

# ─── S3 Paths ───────────────────────────────────────────────────
script_path = "s3://senseai-ml-results-bucket-hj20250426/scripts/generate_explanations.py"
anomaly_scores_s3 = "s3://senseai-ml-results-bucket-hj20250426/inference-results/<LATEST_INFERENCE_JOB>/anomaly_scores.csv"  # Replace this
hscode_ref_s3 = "s3://sagemakerstack-transactionsrawdatabucket6643d2da-q7nfqsc3jpnn/hscode/cleaned_hscode.csv"
output_s3 = f"s3://senseai-ml-results-bucket-hj20250426/explanations/{job_name}/"

# ─── Processor ──────────────────────────────────────────────────
processor = ScriptProcessor(
    image_uri=sagemaker.image_uris.retrieve(
        framework="sklearn",
        region=region,
        version="1.2-1",
        instance_type="ml.m5.4xlarge"
    ),
    role=role,
    instance_count=1,
    instance_type="ml.m5.4xlarge",
    command=["python3"],
    sagemaker_session=session
)

# ─── Run Job ────────────────────────────────────────────────────
processor.run(
    job_name=job_name,
    code=script_path,
    inputs=[
        ProcessingInput(source=anomaly_scores_s3, destination="/opt/ml/processing/input/anomaly_scores.csv"),
        ProcessingInput(source=hscode_ref_s3, destination="/opt/ml/processing/input/cleaned_hscode.csv")
    ],
    outputs=[
        ProcessingOutput(source="/opt/ml/processing/output", destination=output_s3)
    ]
)